# Heterogeneous Ensemble Methods — Voting & Stacking

Unlike **homogeneous ensembles** (e.g., Random Forest — all decision trees), **heterogeneous ensembles** combine **different types of models** to leverage their individual strengths.

---

## Covered in this Notebook:
| Method | Type | How it combines |
|--------|------|-----------------|
| **Voting Classifier** | Heterogeneous | Majority vote / avg probability |
| **Voting Regressor** | Heterogeneous | Averages predictions |
| **Stacking Classifier** | Heterogeneous | Meta-learner on base outputs |
| **Stacking Regressor** | Heterogeneous | Meta-learner on base outputs |

---

## 1. Imports & Data

In [ ]:
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, r2_score

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from sklearn.ensemble import (
    VotingClassifier, VotingRegressor,
    StackingClassifier, StackingRegressor
)

print('✅ All imports ready')

In [ ]:
# Classification dataset
X_c, y_c = make_classification(
    n_samples=500, n_features=20, n_informative=5, n_redundant=2, random_state=42
)
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(X_c, y_c, test_size=0.3, random_state=42)

# Regression dataset
X_r, y_r = make_regression(
    n_samples=500, n_features=20, n_informative=5, random_state=42
)
X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(X_r, y_r, test_size=0.3, random_state=42)

# Base models (shared)
lr  = LogisticRegression(max_iter=1000)
svc = SVC(probability=True)  # probability=True needed for soft voting
dtc = DecisionTreeClassifier(max_depth=3)

lin_reg = LinearRegression()
dtr     = DecisionTreeRegressor(max_depth=3)
svr     = SVR()

print('✅ Datasets and base models created')

## Part A — Voting Classifier

> **Hard Voting**: majority class label wins  
> **Soft Voting**: averages predicted probabilities — usually better, requires `predict_proba`

In [ ]:
voting_clf = VotingClassifier(
    estimators=[('lr', lr), ('svc', svc), ('dtc', dtc)],
    voting='soft'
)

voting_clf.fit(X_c_train, y_c_train)
y_pred = voting_clf.predict(X_c_test)

print(f'Voting Classifier Accuracy: {accuracy_score(y_c_test, y_pred):.4f}')
print(classification_report(y_c_test, y_pred))

## Part B — Voting Regressor

In [ ]:
voting_reg = VotingRegressor(
    estimators=[('lr', lin_reg), ('dtr', dtr), ('svr', svr)]
)

voting_reg.fit(X_r_train, y_r_train)
y_pred_r = voting_reg.predict(X_r_test)

print(f'Voting Regressor R²: {r2_score(y_r_test, y_pred_r):.4f}')

## Part C — Stacking Classifier

**Stacking** uses a **meta-learner** (final estimator) that learns *how to combine* the base model outputs. The base models' predictions become features for the meta-learner.

> More powerful than voting because the meta-learner **learns** the optimal combination.

In [ ]:
stacking_clf = StackingClassifier(
    estimators=[('lr', lr), ('svc', svc), ('dtc', dtc)],
    final_estimator=LogisticRegression(),  # meta-learner
    cv=5                                   # cross-val for generating meta-features
)

stacking_clf.fit(X_c_train, y_c_train)
y_pred = stacking_clf.predict(X_c_test)

print(f'Stacking Classifier Accuracy: {accuracy_score(y_c_test, y_pred):.4f}')
print(classification_report(y_c_test, y_pred))

## Part D — Stacking Regressor

In [ ]:
stacking_reg = StackingRegressor(
    estimators=[('lr', lin_reg), ('dtr', dtr), ('svr', svr)],
    cv=5
)

stacking_reg.fit(X_r_train, y_r_train)

y_pred_test  = stacking_reg.predict(X_r_test)
y_pred_train = stacking_reg.predict(X_r_train)

print(f'Stacking Regressor R² (Test) : {r2_score(y_r_test, y_pred_test):.4f}')
print(f'Stacking Regressor R² (Train): {r2_score(y_r_train, y_pred_train):.4f}')

## Summary

| Method | Pros | Cons |
|--------|------|------|
| Voting | Simple, fast | Equal weight to all models |
| Stacking | Learns optimal combination | Slower, risk of overfitting if not CV'd |

> **Best practice**: Use `cv=5` in stacking to prevent data leakage when generating meta-features.